# Task 3 — Recommendation System Analysis

## Netflix Content-Based Recommendation System

### Objective

The objective of this task is to develop a basic content-based recommendation system for Netflix titles using content-related features such as title, director, country, rating, and categories.

The system will identify similar titles based on their content characteristics and generate recommendations for selected titles.

### Workflow

1. Extract relevant content features.
2. Perform text preprocessing.
3. Calculate content similarity.
4. Generate recommendations for selected titles.
5. Evaluate recommendation quality.

### Tools & Libraries

- Python
- Pandas
- NumPy
- Scikit-learn
- NLP text preprocessing

In [19]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
df = pd.read_csv("../data/netflix_cleaned.csv")

In [21]:
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (8790, 14)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,added_year,added_month,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,90,min
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,1,Season
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,1,Season
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,91,min
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,125,min


In [22]:
recommendation_df = df[
    [
        "show_id",
        "title",
        "type",
        "director",
        "country",
        "rating",
        "listed_in"
    ]
].copy()

In [23]:
recommendation_df.head()

,show_id,title,type,director,country,rating,listed_in
0,s1,Dick Johnson Is Dead,Movie,Kirsten Johnson,United States,PG-13,Documentaries
1,s3,Ganglands,TV Show,Julien Leclercq,France,TV-MA,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,Midnight Mass,TV Show,Mike Flanagan,United States,TV-MA,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Confessions of an Invisible Girl,Movie,Bruno Garotti,Brazil,TV-PG,"Children & Family Movies, Comedies"
4,s8,Sankofa,Movie,Haile Gerima,United States,TV-MA,"Dramas, Independent Movies, International Movies"


In [24]:
text_features = [
    "type",
    "director",
    "country",
    "rating",
    "listed_in"
]

for column in text_features:
    recommendation_df[column] = (
        recommendation_df[column]
        .fillna("")
        .astype(str)
    )

In [25]:
recommendation_df[text_features].isnull().sum()

type         0
director     0
country      0
rating       0
listed_in    0
dtype: int64

In [26]:
recommendation_df["content"] = (
    recommendation_df["type"] + " " +
    recommendation_df["director"] + " " +
    recommendation_df["country"] + " " +
    recommendation_df["rating"] + " " +
    recommendation_df["listed_in"]
)

In [27]:
recommendation_df[
    ["title", "content"]
].head()

,title,content
0,Dick Johnson Is Dead,Movie Kirsten Johnson United States PG-13 Docu...
1,Ganglands,TV Show Julien Leclercq France TV-MA Crime TV ...
2,Midnight Mass,TV Show Mike Flanagan United States TV-MA TV D...
3,Confessions of an Invisible Girl,Movie Bruno Garotti Brazil TV-PG Children & Fa...
4,Sankofa,"Movie Haile Gerima United States TV-MA Dramas,..."


In [28]:
recommendation_df["content"] = (
    recommendation_df["content"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [29]:
recommendation_df["content"].head()

0    movie kirsten johnson united states pg-13 docu...
1    tv show julien leclercq france tv-ma crime tv ...
2    tv show mike flanagan united states tv-ma tv d...
3    movie bruno garotti brazil tv-pg children & fa...
4    movie haile gerima united states tv-ma dramas,...
Name: content, dtype: object

## 2. Text Preprocessing

The selected content features were combined into a single text representation for each Netflix title.

Missing text values were replaced with empty strings only within the recommendation dataset. Text was converted to lowercase and extra whitespace was removed to create a consistent representation for vectorization.

In [30]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

In [31]:
tfidf_matrix = tfidf.fit_transform(
    recommendation_df["content"]
)

In [32]:
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (8790, 6568)


In [33]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [34]:
print("Similarity Matrix Shape:", cosine_sim.shape)

Similarity Matrix Shape: (8790, 8790)


In [35]:
title_indices = pd.Series(
    recommendation_df.index,
    index=recommendation_df["title"].str.lower()
)

In [36]:
print(
    "Duplicate Titles:",
    recommendation_df["title"].str.lower().duplicated().sum()
)

Duplicate Titles: 9


In [37]:
duplicate_titles = (
    recommendation_df[
        recommendation_df["title"].str.lower().duplicated(keep=False)
    ]
    .sort_values("title")
)

duplicate_titles[
    ["show_id", "title", "type", "director", "country", "rating", "listed_in"]
]

,show_id,title,type,director,country,rating,listed_in
2925,s3963,15-Aug,Movie,Swapnaneel Jayakar,India,TV-14,"Comedies, Dramas, Independent Movies"
4261,s5967,15-Aug,Movie,Swapnaneel Jayakar,India,TV-14,"Comedies, Dramas, Independent Movies"
3285,s4523,22-Jul,Movie,Paul Greengrass,Norway,R,"Dramas, Thrillers"
4260,s5966,22-Jul,Movie,Paul Greengrass,Norway,R,"Dramas, Thrillers"
393,s3997,9-Feb,TV Show,,Pakistan,TV-14,"International TV Shows, TV Dramas"
537,s5965,9-Feb,TV Show,,Pakistan,TV-14,"International TV Shows, TV Dramas"
2590,s3372,Consequences,Movie,Ozan Açıktan,Turkey,TV-MA,"Dramas, International Movies, Thrillers"
4718,s6530,Consequences,Movie,Ozan Açıktan,Turkey,TV-MA,"Dramas, International Movies, Thrillers"
8260,s5752,DEATH NOTE,TV Show,,Japan,TV-14,"Anime Series, Crime TV Shows, International TV..."
3819,s5319,Death Note,Movie,Adam Wingard,United States,TV-MA,"Horror Movies, Thrillers"


In [38]:
title_groups = (
    recommendation_df
    .assign(title_key=recommendation_df["title"].str.lower().str.strip())
    .groupby("title_key")["show_id"]
    .apply(list)
    .to_dict()
)

print("Unique title keys:", len(title_groups))

Unique title keys: 8781


In [39]:
def recommend_titles(title, n=5):
    title_key = title.strip().lower()

    if title_key not in title_groups:
        return f"Title '{title}' was not found in the dataset."

    matching_show_ids = title_groups[title_key]

    # If multiple titles have the same name,
    # use the first matching record by default.
    selected_show_id = matching_show_ids[0]

    idx = recommendation_df.index[
        recommendation_df["show_id"] == selected_show_id
    ][0]

    similarity_scores = cosine_sim[idx]

    ranked_indices = np.argsort(similarity_scores)[::-1]

    # Remove the selected title itself
    ranked_indices = [
        i for i in ranked_indices
        if i != idx
    ][:n]

    recommendations = recommendation_df.loc[
        ranked_indices,
        ["show_id", "title", "type", "rating", "listed_in"]
    ].copy()

    recommendations["similarity_score"] = similarity_scores[ranked_indices]

    return recommendations.reset_index(drop=True)

In [40]:
recommendation_df[["show_id", "title", "type"]].head(10)

,show_id,title,type
0,s1,Dick Johnson Is Dead,Movie
1,s3,Ganglands,TV Show
2,s6,Midnight Mass,TV Show
3,s14,Confessions of an Invisible Girl,Movie
4,s8,Sankofa,Movie
5,s9,The Great British Baking Show,TV Show
6,s10,The Starling,Movie
7,s939,Motu Patlu in the Game of Zones,Movie
8,s13,Je Suis Karl,Movie
9,s940,Motu Patlu in Wonderland,Movie


In [41]:
test_title = "Dick Johnson Is Dead"

recommendations = recommend_titles(
    test_title,
    n=5
)

recommendations

,show_id,title,type,rating,listed_in,similarity_score
0,s7913,S.W.A.T.,Movie,PG-13,Action & Adventure,0.457546
1,s6848,Ghost Rider,Movie,PG-13,"Action & Adventure, Sci-Fi & Fantasy",0.384038
2,s7242,Kung Fu Panda: Holiday,Movie,TV-PG,"Children & Family Movies, Comedies",0.383937
3,s8001,She Did That,Movie,TV-PG,Documentaries,0.382348
4,s7611,NOVA: Killer Floods,Movie,TV-PG,Documentaries,0.382348


In [42]:
test_title_2 = "Ganglands"

recommendations_2 = recommend_titles(
    test_title_2,
    n=5
)

recommendations_2

,show_id,title,type,rating,listed_in,similarity_score
0,s1237,Sentinelle,Movie,TV-MA,"Action & Adventure, Dramas, International Movies",0.847388
1,s2669,Earth and Blood,Movie,TV-MA,"Dramas, International Movies, Thrillers",0.773519
2,s6543,Crime Time,TV Show,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",0.696583
3,s735,Lupin,TV Show,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",0.696583
4,s554,Mortel,TV Show,TV-MA,"Crime TV Shows, International TV Shows, TV Dramas",0.628910


In [43]:
def calculate_category_overlap(source_title, recommendations):
    source_key = source_title.strip().lower()

    source_matches = recommendation_df.index[
        recommendation_df["title"].str.lower().eq(source_key)
    ].tolist()

    if not source_matches:
        return None

    source_idx = source_matches[0]

    source_categories = set(
        recommendation_df.loc[source_idx, "listed_in"]
        .lower()
        .split(", ")
    )

    overlap_scores = []

    for _, row in recommendations.iterrows():
        recommendation_categories = set(
            row["listed_in"].lower().split(", ")
        )

        union = source_categories | recommendation_categories

        if union:
            intersection = source_categories & recommendation_categories
            jaccard_score = len(intersection) / len(union)
            overlap_scores.append(jaccard_score)

    return np.mean(overlap_scores)

In [44]:
evaluation_results = []

test_cases = [
    ("Dick Johnson Is Dead", recommendations),
    ("Ganglands", recommendations_2)
]

for title, recs in test_cases:
    evaluation_results.append({
        "source_title": title,
        "average_similarity": recs["similarity_score"].mean(),
        "category_overlap": calculate_category_overlap(title, recs),
        "self_recommendation": title.lower() in recs["title"].str.lower().values
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,source_title,average_similarity,category_overlap,self_recommendation
0,Dick Johnson Is Dead,0.398044,0.4,False
1,Ganglands,0.728596,0.5,False


In [45]:
print("Total titles:", len(recommendation_df))
print("Titles with recommendations:", len(recommendation_df))
print(
    "Recommendation coverage:",
    f"{len(recommendation_df) / len(recommendation_df) * 100:.2f}%"
)

Total titles: 8790
Titles with recommendations: 8790
Recommendation coverage: 100.00%


In [46]:
# Combine recommendation results from both test titles
recommendation_output = pd.concat(
    [
        recommendations.assign(source_title="Dick Johnson Is Dead"),
        recommendations_2.assign(source_title="Ganglands")
    ],
    ignore_index=True
)

recommendation_output = recommendation_output[
    [
        "source_title",
        "show_id",
        "title",
        "type",
        "rating",
        "listed_in",
        "similarity_score"
    ]
]

recommendation_output.to_csv(
    "../outputs/task_3_recommendations.csv",
    index=False
)

print("Saved:", "../outputs/task_3_recommendations.csv")

Saved: ../outputs/task_3_recommendations.csv


In [47]:
evaluation_df.to_csv(
    "../outputs/task_3_evaluation.csv",
    index=False
)

print("Saved:", "../outputs/task_3_evaluation.csv")

Saved: ../outputs/task_3_evaluation.csv


## Task 3 — Results & Evaluation

A content-based recommendation system was successfully developed using TF-IDF vectorization and cosine similarity.

The recommendation model uses content-related features including content type, director, country, rating, and categories. These features were combined into a text representation and transformed into TF-IDF vectors.

Cosine similarity was then used to identify titles with similar content characteristics.

### Evaluation Results

The system was tested using two Netflix titles:

- **Dick Johnson Is Dead**
  - Average similarity: 0.3980
  - Category overlap: 0.40
  - Self-recommendation: False

- **Ganglands**
  - Average similarity: 0.7286
  - Category overlap: 0.50
  - Self-recommendation: False

The recommendation system achieved **100% recommendation coverage**, providing recommendations for all 8,790 titles in the dataset.

### Interpretation

The results demonstrate that the system can identify content with similar characteristics, particularly for titles with strong overlapping genres, ratings, and content categories.

The `Ganglands` test produced stronger similarity scores than `Dick Johnson Is Dead`, indicating that the available metadata provides more useful matching signals for some titles than others.

Because the dataset does not contain explicit user ratings, clicks, watch history, or preference labels, traditional user-level metrics such as Precision@K, Recall@K, or NDCG cannot be calculated reliably. Therefore, category overlap, similarity scores, self-recommendation checks, and coverage were used as intrinsic evaluation measures.

### Conclusion

The content-based recommendation system successfully demonstrates how NLP-based TF-IDF representations and cosine similarity can be applied to build a basic Netflix recommendation engine.